# NeuroLens: Complete Research Training Pipeline

**Research question:** can a shared representation space connect a 2D brain MRI slice to the 3D MRI volume from the same patient while supporting tumour classification and segmentation?

This notebook is built around the repository implementation. BraTS supplies genuine same-patient 2D/3D pairs and voxel labels. BRISC supplies independent 2D T1 classification and segmentation records; it is audited and inspected separately and never treated as paired with BraTS. The checked-in architecture, losses, dataset preprocessing, and saved patient splits remain the source of truth.

The known integration evidence is documented rather than rerun here: 371 complete BraTS cases (296/37/38 saved patients), four MRI modalities, 64-cubed real batch tensors, finite loss, and 108/108 trainable tensors receiving finite gradients in the earlier gradient check. BRISC's earlier audit found 6,000 classification images and 4,793 valid segmentation pairs with zero classification/segmentation image-content overlap. Runtime cells below verify current Drive availability and report current data-derived values.

## 1. Introduction and research methodology

### Research workflow

```mermaid
flowchart TD
  A[Mount Drive and inspect repository] --> B[Read-only BRISC and BraTS audits]
  B --> C[Verify saved patient-level manifests]
  C --> D[Normalize and crop BraTS modalities; keep labels aligned]
  D --> E[Same-patient 2D slice + 3D volume DataLoader]
  E --> F[Existing NeuroLens 2D and 3D encoders]
  F --> G[Binary tumour classification + 4-class segmentation]
  F --> H[Shared projected 2D/3D embeddings]
  G --> I[Condition A multitask optimization]
  I --> J[Validation selects best checkpoint]
  J --> K[Held-out test evaluation and research report]
  R[Independent 2D BRISC tasks] -. no cross-dimensional pairing .-> B
```

**Methodology:** a 2D encoder summarizes a slice and a 3D encoder summarizes the corresponding patient crop. Projection heads map both features into the common embedding dimension. The 3D decoder produces voxel logits. Condition A is the controlled baseline; according to the trainer it optimizes classification and segmentation terms, with contrastive and consistency weights disabled. Alignment metrics may still be reported on genuine paired BraTS embeddings, but they are not Condition A training objectives.

## 2. Colab environment setup

Mount Drive, locate the repository in a small set of known locations (or clone it only when absent), check the existing Python/PyTorch/CUDA stack, install only `requirements.txt`, and seed all supported RNGs. GPU use is recommended for the 3D path; CPU remains usable for setup and read-only inspection.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util, platform, random

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running inside Colab; Drive mounting is unavailable in this runtime.")

DRIVE_PROJECT = Path("/content/drive/MyDrive/NeuroLens")
REPO_CANDIDATES = [DRIVE_PROJECT, Path("/content/NeuroLens"), Path.cwd()]
REPO = next((p for p in REPO_CANDIDATES if (p / "config.yaml").is_file() and (p / "neurolens").is_dir()), None)
if REPO is None:
    REPO = Path("/content/NeuroLens")
    if not Path("/content").exists():
        raise RuntimeError("Expected Colab /content directory or an existing NeuroLens checkout.")
    subprocess.run(["git", "clone", "https://github.com/MSSG-07/Neurolens.git", str(REPO)], check=True)
os.chdir(REPO)
print("Repository:", REPO, "| Python:", sys.version.split()[0], "| Platform:", platform.platform())

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is not installed in this runtime. Select a standard Colab runtime; this notebook does not force-install or replace PyTorch.")
import torch
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    print("WARNING: no CUDA GPU detected. Inspection is fine; the training cell requires CUDA by default.")

requirements = REPO / "requirements.txt"
if not requirements.is_file():
    raise FileNotFoundError(f"Project requirements file is missing: {requirements}")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
from neurolens.utils.reproducibility import seed_everything
SEED = 42
seed_everything(SEED)
print("Seed configured:", SEED, "| deterministic cuDNN:", torch.backends.cudnn.deterministic)


## 3. Repository and implementation inspection

The source is inspected from this checkout, not paraphrased from a generic model template. The output lists the package files and prints the configuration and principal entry points so configuration drift is visible before data access.

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown

for path in sorted((REPO / "neurolens").rglob("*.py")):
    print(path.relative_to(REPO))
for relative in ["config.yaml", "train.py", "evaluate.py", "neurolens/training/train.py",
                 "neurolens/datasets/brats.py", "neurolens/datasets/transforms.py",
                 "neurolens/models/multitask.py", "neurolens/losses/multitask_loss.py",
                 "neurolens/training/trainer.py", "neurolens/training/checkpoint.py"]:
    print(f"\n{'='*18} {relative} {'='*18}\n")
    print((REPO / relative).read_text(encoding="utf-8"))


## 4. Dataset organization and read-only auditing

The paths below are the known Drive locations. The notebook checks rather than creates dataset files. BRISC resolution uses the repository `BRISCPathResolver`, which checks the manifest relative path and the known canonical/`(1)` directory pair. A complete content-hash audit is available behind an explicit toggle because it reads thousands of files and has already been completed in the prior integration work.

In [ ]:
from pathlib import Path
import csv, json, collections

BRATS_ROOT = DRIVE_PROJECT / "data/raw/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
BRISC_ROOT = DRIVE_PROJECT / "data/raw/BRISC2025/brisc2025"
MANIFEST_DIR = DRIVE_PROJECT / "manifests"
TRAIN_MANIFEST = MANIFEST_DIR / "train.csv"
VAL_MANIFEST = MANIFEST_DIR / "validation.csv"
TEST_MANIFEST = MANIFEST_DIR / "test.csv"
BRISC_MANIFEST = BRISC_ROOT / "manifest.csv"

required_paths = {"BraTS root": BRATS_ROOT, "BRISC root": BRISC_ROOT,
                  "BraTS train manifest": TRAIN_MANIFEST, "BraTS validation manifest": VAL_MANIFEST,
                  "BraTS test manifest": TEST_MANIFEST, "BRISC manifest": BRISC_MANIFEST}
for label, path in required_paths.items():
    print(f"{label}: {'OK' if path.exists() else 'MISSING'} — {path}")
missing = [f"{name}: {path}" for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError("Required Drive inputs are unavailable. Mount the expected Drive and verify these paths:\n" + "\n".join(missing))

print("\nBRISC task-directory variants:")
for name in ("classification_task", "classification_task (1)", "segmentation_task", "segmentation_task (1)"):
    path = BRISC_ROOT / name
    print(name, "exists=", path.is_dir(), "top-level entries=", len(list(path.iterdir())) if path.is_dir() else 0)

with BRISC_MANIFEST.open(newline="", encoding="utf-8-sig") as f:
    brisc_rows = list(csv.DictReader(f))
if not brisc_rows:
    raise ValueError("BRISC manifest is empty")
print("\nBRISC manifest rows:", len(brisc_rows), "columns:", sorted(brisc_rows[0]))
split_key = next((k for k in ("split", "subset", "partition") if k in brisc_rows[0]), None)
if split_key is None:
    raise ValueError("BRISC manifest has no explicit split column; refusing to invent a split")
print("Split counts:", dict(collections.Counter(r[split_key].strip().casefold() for r in brisc_rows)))
print("Tumor-label counts (all manifest records):", dict(collections.Counter(r.get("tumor_label", "").strip().casefold() for r in brisc_rows)))
print("Mask flags:", dict(collections.Counter(r.get("is_mask", "") for r in brisc_rows)))
brisc_json = BRISC_ROOT / "manifest.json"
if brisc_json.is_file():
    manifest_json_data = json.loads(brisc_json.read_text(encoding="utf-8"))
    print("manifest.json top-level type/keys:", type(manifest_json_data).__name__, list(manifest_json_data)[:20] if isinstance(manifest_json_data, dict) else len(manifest_json_data))
else:
    print("manifest.json is absent; CSV remains the dataset loader's manifest input.")
import hashlib
for manifest_file in (BRISC_MANIFEST, brisc_json):
    checksum_candidates = (manifest_file.with_suffix(manifest_file.suffix + ".sha256"), manifest_file.with_suffix(".sha256"))
    checksum_file = next((p for p in checksum_candidates if p.is_file()), None)
    if checksum_file:
        expected_hash = checksum_file.read_text(encoding="utf-8").split()[0]
        actual_hash = hashlib.sha256(manifest_file.read_bytes()).hexdigest()
        print(manifest_file.name, "checksum matches:", expected_hash.casefold() == actual_hash.casefold())
    else:
        print(manifest_file.name, "checksum file: not present")
print("BRISC audit facts from prior completed integration: 6,000 classification images; 4,793 segmentation images/masks; 4,793 valid pairs; 15,586 manifest rows; zero classification/segmentation image-content overlap.")

from neurolens.datasets.brisc import BRISCDataset, BRISCPathResolver, validate_brisc
brisc_resolver = BRISCPathResolver(BRISC_ROOT)
example_relative = next((r.get("relative_path", "") for r in brisc_rows if r.get("relative_path")), None)
if example_relative:
    print("Resolver example:", example_relative, "->", brisc_resolver.resolve(example_relative))
RUN_FULL_BRISC_AUDIT = False  # Set True to run the repository's full decode/hash/pair audit on Drive.
if RUN_FULL_BRISC_AUDIT:
    brisc_audit = validate_brisc(BRISC_ROOT, BRISC_MANIFEST)
    print(json.dumps({k: v for k, v in brisc_audit.items() if k not in {"missing_files", "ambiguous_files", "corrupt_files", "mismatched_masks", "duplicate_references", "duplicate_image_content", "duplicate_mask_content", "trace_rows", "errors"}}, indent=2, default=str))
    print("Audit detail counts:", {k: len(brisc_audit.get(k, [])) for k in ("missing_files", "ambiguous_files", "corrupt_files", "mismatched_masks", "duplicate_references", "duplicate_image_content", "duplicate_mask_content", "errors")})
    if any(brisc_audit.get(k) for k in ("missing_files", "ambiguous_files", "corrupt_files", "mismatched_masks", "errors")):
        raise RuntimeError("Full BRISC audit reported invalid records; inspect brisc_audit before using this dataset.")


### BRISC task responsibilities

`BRISCDataset(root, manifest, task, split)` reads official manifest records and the split explicitly. Classification labels use the loader’s named mapping (`glioma`, `meningioma`, `pituitary`, `no_tumor`); segmentation masks are returned as masks. Prior loader checks produced one-channel 512×512 images and 512×512 masks. Since the checked-in NeuroLens model expects four input channels and a binary class head, BRISC is not fed to that model in this baseline. Adapting it would require a separate, explicitly designed experiment and must not create any BraTS correspondence.

In [ ]:
from torch.utils.data import DataLoader
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

brisc_splits = sorted({r[split_key].strip().casefold() for r in brisc_rows})
print("Manifest-defined BRISC splits:", brisc_splits)
if not {"train", "test"}.issubset(set(brisc_splits)):
    print("WARNING: this manifest does not expose both literal train and test names; no split is being synthesized.")
else:
    brisc_cls_train = BRISCDataset(BRISC_ROOT, BRISC_MANIFEST, task="classification", split="train")
    brisc_seg_train = BRISCDataset(BRISC_ROOT, BRISC_MANIFEST, task="segmentation", split="train")
    print("BRISC train datasets:", len(brisc_cls_train), "classification;", len(brisc_seg_train), "segmentation")
    if "test" in brisc_splits:
        brisc_cls_test = BRISCDataset(BRISC_ROOT, BRISC_MANIFEST, task="classification", split="test")
        brisc_seg_test = BRISCDataset(BRISC_ROOT, BRISC_MANIFEST, task="segmentation", split="test")
        print("Official BRISC test datasets (inspection only):", len(brisc_cls_test), "classification;", len(brisc_seg_test), "segmentation")
    brisc_cls_batch = next(iter(DataLoader(brisc_cls_train, batch_size=4, shuffle=False, num_workers=0)))
    brisc_seg_batch = next(iter(DataLoader(brisc_seg_train, batch_size=4, shuffle=False, num_workers=0)))
    print("BRISC classification batch shapes:", tuple(brisc_cls_batch["image"].shape), tuple(brisc_cls_batch["class_label"].shape))
    print("BRISC segmentation batch shapes:", tuple(brisc_seg_batch["image"].shape), tuple(brisc_seg_batch["mask"].shape))
    if len(brisc_cls_train):
        brisc_cls_sample = brisc_cls_train[0]
        bx, by = brisc_cls_sample["image"], brisc_cls_sample["class_label"]
        print("Classification sample:", tuple(bx.shape), bx.dtype, int(by), "=>", brisc_cls_sample["tumor_label"], "path:", brisc_cls_sample["image_path"])
        plt.figure(figsize=(5, 5)); plt.imshow(bx.squeeze().numpy(), cmap="gray"); plt.title("BRISC train T1 sample"); plt.axis("off"); plt.show()
    if len(brisc_seg_train):
        brisc_seg_sample = brisc_seg_train[0]
        bx, bm = brisc_seg_sample["image"], brisc_seg_sample["mask"]
        print("Segmentation sample:", tuple(bx.shape), bx.dtype, tuple(bm.shape), bm.dtype, "mask values:", torch.unique(bm).tolist(), "image/mask paths:", brisc_seg_sample["image_path"], brisc_seg_sample["mask_path"])
        fig, ax = plt.subplots(1, 2, figsize=(9, 4)); ax[0].imshow(bx.squeeze().numpy(), cmap="gray"); ax[0].set_title("BRISC image"); ax[1].imshow(bm.squeeze().numpy(), cmap="viridis"); ax[1].set_title("Manifest-linked mask"); [a.axis("off") for a in ax]; plt.show()
    print("Do not append BRISC items to BraTS batches: they do not have 3D patient partners.")


## 5. BraTS saved splits and leakage prevention

The saved patient lists are loaded by `read_patient_ids`; `validate_saved_splits` rejects intersections. These files are not regenerated or edited. Patient-level separation keeps slices, patches, and correlated anatomy from one patient out of other partitions. Validation controls checkpoint selection; test stays unopened until the final evaluation section.

In [ ]:
from neurolens.config import load_config
from neurolens.training.train import validate_saved_splits, make_datasets
from neurolens.datasets.manifests import read_patient_ids

cfg = load_config(str(REPO / "config.yaml"))
cfg.brats_root = str(BRATS_ROOT)
cfg.data_root = str(BRATS_ROOT)
cfg.manifest_train, cfg.manifest_val, cfg.manifest_test = map(str, (TRAIN_MANIFEST, VAL_MANIFEST, TEST_MANIFEST))
cfg.output_dir = str(DRIVE_PROJECT / "runs/baseline_A")
cfg.alignment_condition = "A"
cfg.seed = SEED
cfg.validate()
split_counts = validate_saved_splits(cfg)
split_ids = {name: set(read_patient_ids(str(path), cfg.manifest_id_column)) for name, path in
             (("train", TRAIN_MANIFEST), ("validation", VAL_MANIFEST), ("test", TEST_MANIFEST))}
print("Saved split patient counts:", split_counts)
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    overlap = split_ids[left] & split_ids[right]
    print(f"{left}/{right} overlap:", len(overlap))
    assert not overlap, f"Patient leakage: {left}/{right} share {sorted(overlap)[:5]}"
assert split_counts == {"train": 296, "validation": 37, "test": 38}, f"Unexpected saved split counts: {split_counts}"
print("Split verification passed; the test IDs have not been used to create any training or validation data.")


## 6. MRI preprocessing and spatial consistency

`BraTSPairedDataset` loads each patient’s NIfTI modalities in `cfg.modalities` order (`t1n`, `t1c`, `t2w`, `t2f`) and the segmentation. It transposes NIfTI `(X,Y,Z)` arrays to `(Z,Y,X)`, applies per-channel nonzero-voxel z-score normalization, center-crops/pads images and labels to `patch_size`, and maps source label 4 to contiguous label 3. The 2D slice is sampled from the same cropped volume along `slice_axis`; the label remains aligned with the 3D volume. No geometric augmentation is implemented in the current baseline. Do not apply ordinary intensity normalization or interpolating transforms to label masks.

In [ ]:
from neurolens.datasets.brats import discover_patients, cases_from_manifest, BraTSPairedDataset
from neurolens.datasets.transforms import normalize_modalities, center_crop_or_pad

cases = discover_patients(cfg.brats_root, cfg.modalities)
print("Complete discovered BraTS cases:", len(cases))
train_ids = read_patient_ids(cfg.manifest_train, cfg.manifest_id_column)
train_cases = cases_from_manifest(cases, train_ids, "train")
train_dataset = BraTSPairedDataset(cfg.brats_root, train_cases, cfg.modalities, cfg.patch_size, cfg.slice_axis)
sample = train_dataset[0]
print("Patient:", sample["patient_id"], "2D:", tuple(sample["image2d"].shape), "3D:", tuple(sample["image3d"].shape),
      "seg:", tuple(sample["segmentation"].shape), "labels:", torch.unique(sample["segmentation"]).tolist(),
      "slice index:", sample["slice_index"], "class label:", int(sample["class_label"]))
assert sample["image2d"].shape[0] == len(cfg.modalities)
assert tuple(sample["image3d"].shape[1:]) == tuple(cfg.patch_size)
assert sample["segmentation"].shape == sample["image3d"].shape[1:]
assert set(torch.unique(sample["segmentation"]).tolist()).issubset({0, 1, 2, 3})

# Paired image and label views use the exact same slice index and axis.
volume_np = sample["image3d"].numpy()
label_np = sample["segmentation"].numpy()
slice_img = sample["image2d"].numpy()
slice_mask = np.take(label_np, sample["slice_index"], axis=cfg.slice_axis)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(slice_img[0], cmap="gray"); axes[0].set_title("2D sampled t1n slice")
axes[1].imshow(slice_img[0], cmap="gray"); axes[1].imshow(np.ma.masked_where(slice_mask == 0, slice_mask), alpha=.45, vmin=1, vmax=3, cmap="viridis"); axes[1].set_title("Same-plane label overlay")
axes[2].imshow(volume_np[0, volume_np.shape[1]//2], cmap="gray"); axes[2].set_title("3D volume middle depth (t1n)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()


## 7. BraTS DataLoaders and batch inspection

The trainer’s batches contain same-patient `image2d`, `image3d`, `segmentation`, and `class_label` values. Training shuffles patients; validation and test are ordered. No patient in validation or test can contribute a training slice. This cell constructs the train and validation loaders only; the test loader is deliberately deferred.

In [ ]:
import torch
from torch.utils.data import DataLoader

train_dataset, val_dataset = make_datasets(cfg)
device = torch.device(cfg.device if cfg.device == "cpu" or torch.cuda.is_available() else "cpu")
pin = device.type == "cuda"
train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers,
                          pin_memory=pin, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers,
                        pin_memory=pin, drop_last=False)
batch = next(iter(train_loader))
print("Device:", device, "| train/validation patients:", len(train_dataset), len(val_dataset))
for key, value in batch.items():
    if torch.is_tensor(value): print(f"{key}: shape={tuple(value.shape)} dtype={value.dtype} device={value.device}")
    elif isinstance(value, (list, tuple)): print(f"{key}: {value[:4]}")
print("Class labels in batch:", dict(zip(*np.unique(batch["class_label"].numpy(), return_counts=True))))
print("Segmentation labels in batch:", torch.unique(batch["segmentation"]).tolist())


## 8. Existing NeuroLens architecture

`NeuroLens` constructs `Encoder2D`, `Encoder3D`, and separate projection heads using the checked-in constructor. The 2D encoder is a sequence of strided convolution, depthwise convolution, pointwise convolution, batch normalization, and ReLU blocks followed by global pooling, a feature projection, and binary classifier. The 3D encoder uses four convolutional stages with pooling, a volume projection, and a decoder with transposed convolutions and skip connections to produce four segmentation logits per voxel. Projection heads map both pathway embeddings into the same configured dimension and L2-normalize them. The forward result keys are `classification_logits`, `embedding2d`, `embedding3d`, `segmentation_logits`, `features2d`, `pooled2d`, and `pooled3d`.

### Architecture diagram (implementation-faithful)

```mermaid
flowchart LR
  X2[Same-patient 2D slice\nB×4×H×W] --> E2[Encoder2D\nstrided/depthwise/pointwise CNN]
  E2 --> C[Binary classifier\nB×2 logits]
  E2 --> P2[2D projection head\nB×256 normalized]
  X3[Same-patient 3D crop\nB×4×D×H×W] --> E3[Encoder3D\n4 stages + pooling]
  E3 --> P3[3D projection head\nB×256 normalized]
  E3 --> U[3D decoder\nupsampling + skip connections]
  U --> S[4-class voxel logits\nB×4×D×H×W]
  P2 -. shared embedding space .- P3
  C --> L[MultiTaskLoss]
  S --> L
  P2 --> L
  P3 --> L
```

The diagonal pairing is the dataset invariant: row *i* in `image2d` and `image3d` belongs to the same BraTS patient. BRISC rows never enter these paired batches.

In [ ]:
from neurolens.models.multitask import NeuroLens
architecture_model = NeuroLens(len(cfg.modalities), cfg.base_channels, cfg.projection_dim, cfg.num_classes, cfg.classification_classes)
print(architecture_model)
print("Trainable parameters:", sum(p.numel() for p in architecture_model.parameters() if p.requires_grad))
print("Configured output shapes for the inspected batch:", {"classification_logits": (cfg.batch_size, cfg.classification_classes), "embedding2d/embedding3d": (cfg.batch_size, cfg.projection_dim), "segmentation_logits": (cfg.batch_size, cfg.num_classes, *cfg.patch_size)})
print("The previously completed real-data forward/gradient checks are not repeated in this inspection cell.")


## 9. Loss functions and optimization

`MultiTaskLoss` uses cross-entropy for binary class logits, multiclass segmentation cross-entropy plus a soft Dice loss averaged over all four classes (including background), cosine embedding consistency, and symmetric in-batch InfoNCE. `Trainer` weights classification and segmentation from config; for Condition A it explicitly sets contrastive and consistency weights to zero. Thus the Condition A optimized objective is `1.0 × classification + 1.0 × segmentation`. The class target is tumour presence, not tumour subtype. Source segmentation labels are mapped to `0=background, 1=NCR, 2=ED, 3=ET`.

The optimizer in `Trainer` is AdamW with the config learning rate and weight decay. Gradient accumulation is applied over four steps; AMP uses a CUDA GradScaler when configured and available. The repository has no scheduler, gradient clipping, early stopping, or built-in best-checkpoint selection. A final small validation batch with one row can make InfoNCE undefined, but it is omitted for singleton batches; in Condition A its weight is zero anyway.

In [ ]:
from neurolens.losses.multitask_loss import MultiTaskLoss
effective_weights = {"classification": cfg.lambda_classification,
                     "segmentation": cfg.lambda_segmentation,
                     "contrastive": 0.0, "consistency": 0.0}
criterion = MultiTaskLoss(effective_weights, cfg.temperature)
print("Condition A loss weights:", effective_weights)
print("MultiTaskLoss implementation:", criterion.__class__.__module__ + "." + criterion.__class__.__name__)
print("Earlier real-batch forward/loss validation is recorded in the introduction; this notebook does not rerun that smoke test.")
print("AdamW:", {"learning_rate": cfg.learning_rate, "weight_decay": cfg.weight_decay})
print("batch_size:", cfg.batch_size, "accumulation_steps:", cfg.accumulation_steps, "epochs:", cfg.epochs,
      "AMP config:", cfg.mixed_precision, "scheduler: none", "gradient clipping: none")


## 10. Condition A configuration and explicit training gate

This is the only cell that starts a training loop, and it is disabled by default. Review paths and storage first; then explicitly set `START_TRAINING = True` and execute this cell. The baseline uses `cfg` from `config.yaml` with only runtime dataset/manifest/output paths and Condition A set. Training writes under `MyDrive/NeuroLens/runs/baseline_A`. The saved test manifest is not passed to the trainer. The code reuses `Trainer.run_epoch`, its AdamW, loss, mixed precision, and accumulation behavior. No full training is run while constructing this notebook.

## 11. Checkpointing and recovery

The repository checkpoint helper atomically saves model, optimizer, scaler, epoch, config, and Torch CPU RNG state. The notebook additionally saves training history, best validation loss/epoch, Python RNG, NumPy RNG, and CUDA RNG states in `training_state.pt`. `latest.pt` updates each epoch; `best_validation.pt` updates only when validation total loss improves. Resume restores those states and continues from the next epoch. There is no scheduler state because the repository has no scheduler.

In [ ]:
import copy, json, math, random, shutil, time
from dataclasses import asdict
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader
from neurolens.training.trainer import Trainer
from neurolens.training.checkpoint import save_checkpoint, load_checkpoint
from neurolens.utils.logging import CSVLogger
from neurolens.utils.metrics import classification_metrics
from neurolens.evaluation.segmentation import segmentation_metrics
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda values, **kwargs: values

START_TRAINING = False  # Change to True only when intentionally starting/resuming the experiment.
RESUME = False          # Set True to restore latest.pt and training_state.pt.
REQUIRE_GPU = True
RUN_DIR = Path(cfg.output_dir)
if not START_TRAINING:
    print("Training is gated. Set START_TRAINING=True and rerun this cell to begin Condition A.")
else:
    if REQUIRE_GPU and not torch.cuda.is_available():
        raise RuntimeError("Condition A training requires a CUDA GPU in this notebook. Select a GPU runtime; no CPU fallback is started.")
    if RUN_DIR.exists() and not RESUME and any(RUN_DIR.iterdir()):
        raise FileExistsError(f"Run directory is not empty: {RUN_DIR}. Choose a fresh output directory or set RESUME=True to continue the saved run.")
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    cfg.device = "cuda" if torch.cuda.is_available() else "cpu"
    cfg.alignment_condition = "A"
    cfg.validate()
    config_record = asdict(cfg)
    config_json = json.loads(json.dumps(config_record, default=list))
    if RESUME:
        saved_config_path = RUN_DIR / "config.json"
        if not saved_config_path.is_file(): raise FileNotFoundError("Resume requested but config.json is missing.")
        if json.loads(saved_config_path.read_text(encoding="utf-8")) != config_json:
            raise ValueError("Current experiment config differs from saved config.json; refusing an unsafe resume.")
    (RUN_DIR / "config.json").write_text(json.dumps(config_record, indent=2, default=list), encoding="utf-8")
    import yaml
    (RUN_DIR / "runtime_config.yaml").write_text(yaml.safe_dump(config_json, sort_keys=False), encoding="utf-8")
    model = NeuroLens(len(cfg.modalities), cfg.base_channels, cfg.projection_dim, cfg.num_classes, cfg.classification_classes)
    trainer = Trainer(model, cfg)
    trainer.logger = CSVLogger(RUN_DIR / "training.csv")
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers,
                              pin_memory=trainer.device.type == "cuda", drop_last=False)
    val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers,
                            pin_memory=trainer.device.type == "cuda", drop_last=False)
    history, best_val, best_epoch, start_epoch, elapsed_before = [], float("inf"), None, 1, 0.0
    state_path = RUN_DIR / "training_state.pt"
    if RESUME:
        latest_path = RUN_DIR / "latest.pt"
        if not latest_path.is_file() or not state_path.is_file():
            raise FileNotFoundError("Resume requested, but latest.pt or training_state.pt is missing.")
        last_epoch = load_checkpoint(latest_path, trainer.model, trainer.optimizer, trainer.scaler, map_location=trainer.device)
        extra = torch.load(state_path, map_location="cpu", weights_only=False)
        history, best_val, best_epoch = extra["history"], extra["best_val"], extra["best_epoch"]
        elapsed_before = float(extra.get("runtime_seconds", 0.0))
        random.setstate(extra["python_rng"]); np.random.set_state(extra["numpy_rng"])
        if torch.cuda.is_available() and extra.get("cuda_rng") is not None: torch.cuda.set_rng_state_all(extra["cuda_rng"])
        start_epoch = last_epoch + 1
        print(f"Resumed after epoch {last_epoch}; best validation total={best_val:.6f} at epoch {best_epoch}.")
    run_started = time.time()
    def collect_validation_metrics(model, loader, device):
        model.eval(); y_true, y_pred, y_prob, seg_rows = [], [], [], []
        with torch.inference_mode():
            for vb in loader:
                vo = model(vb["image2d"].to(device).float(), vb["image3d"].to(device).float())
                prob = vo["classification_logits"].softmax(1).cpu().numpy(); pred = prob.argmax(1)
                y_true.extend(vb["class_label"].tolist()); y_pred.extend(pred.tolist()); y_prob.extend(prob.tolist())
                seg = vo["segmentation_logits"].argmax(1).cpu().numpy(); truth = vb["segmentation"].cpu().numpy()
                seg_rows.extend(segmentation_metrics(seg[j], truth[j]) for j in range(len(seg)))
        cls = classification_metrics(y_true, y_pred, y_prob)
        return {"accuracy": cls["accuracy"], "precision": cls["precision"], "recall": cls["recall"], "f1": cls["f1"],
                "seg_dice": float(np.nanmean([r["dice"] for r in seg_rows])),
                "seg_iou": float(np.nanmean([r["iou"] for r in seg_rows]))}
    for epoch in tqdm(range(start_epoch, cfg.epochs + 1), desc="Condition A epochs"):
        train_metrics = trainer.run_epoch(train_loader, training=True)
        val_metrics = trainer.run_epoch(val_loader, training=False)
        val_scores = collect_validation_metrics(trainer.model, val_loader, trainer.device)
        row = {"epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()},
               **{f"val_{k}": v for k, v in val_metrics.items()}, **{f"val_{k}": v for k, v in val_scores.items()}}
        if not all(math.isfinite(float(v)) for v in row.values()):
            raise FloatingPointError(f"Non-finite epoch metric; refusing to checkpoint: {row}")
        trainer.logger.log(row); history.append(row)
        latest_path = RUN_DIR / "latest.pt"
        save_checkpoint(latest_path, trainer.model, trainer.optimizer, trainer.scaler, epoch, config_record)
        improved = val_metrics.get("total", float("inf")) < best_val
        if improved:
            best_val, best_epoch = val_metrics["total"], epoch
            save_checkpoint(RUN_DIR / "best_validation.pt", trainer.model, trainer.optimizer, trainer.scaler, epoch, config_record)
        torch.save({"history": history, "best_val": best_val, "best_epoch": best_epoch,
                    "python_rng": random.getstate(), "numpy_rng": np.random.get_state(),
                    "runtime_seconds": elapsed_before + time.time() - run_started,
                    "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}, state_path)
        print(json.dumps(row), "| best_val=", best_val, "best_epoch=", best_epoch,
              "| GPU peak GiB=", round(torch.cuda.max_memory_allocated() / 2**30, 3) if torch.cuda.is_available() else None)
    metadata = {"seed": cfg.seed, "device": str(trainer.device), "condition": "A",
                "runtime_seconds_total": elapsed_before + time.time() - run_started,
                "total_epochs_recorded": len(history), "best_epoch": best_epoch, "best_validation_total": best_val,
                "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
    (RUN_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print("Training session complete. latest:", latest_path, "best:", RUN_DIR / "best_validation.pt")


## 12. Training curves

Curves are generated from the saved epoch CSV when training has produced it. Training and validation values are means of per-batch loss components, matching the repository trainer aggregation. There is no learning-rate curve because no scheduler is configured.

In [ ]:
import csv
import matplotlib.pyplot as plt

TRAINING_CSV = Path(cfg.output_dir) / "training.csv"
if TRAINING_CSV.is_file():
    with TRAINING_CSV.open(newline="", encoding="utf-8") as f: rows = list(csv.DictReader(f))
    frame = {key: [float(row[key]) for row in rows] for key in rows[0]} if rows else {}
    if len(frame):
        plot_dir = Path(cfg.output_dir) / "plots"; plot_dir.mkdir(parents=True, exist_ok=True)
        metrics = [c.removeprefix("train_") for c in frame if c.startswith("train_")]
        for metric in metrics:
            val_col = f"val_{metric}"
            if val_col not in frame: continue
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.plot(frame["epoch"], frame[f"train_{metric}"], label=f"Train {metric}")
            ax.plot(frame["epoch"], frame[val_col], label=f"Validation {metric}")
            ax.set(xlabel="Epoch", ylabel=metric.replace("_", " "), title=f"Condition A: {metric}"); ax.legend(); ax.grid(alpha=.25)
            fig.tight_layout(); fig.savefig(plot_dir / f"{metric}.png", dpi=150); plt.show()
        for metric in ("accuracy", "f1", "seg_dice", "seg_iou"):
            column = f"val_{metric}"
            if column not in frame: continue
            fig, ax = plt.subplots(figsize=(7, 4)); ax.plot(frame["epoch"], frame[column], marker="o")
            ax.set(xlabel="Epoch", ylabel=metric, title=f"Validation {metric}"); ax.grid(alpha=.25); fig.tight_layout()
            fig.savefig(plot_dir / f"validation_{metric}.png", dpi=150); plt.show()
        print("Saved curves under", plot_dir)
    else: print("Training CSV exists but has no epoch records yet.")
else:
    print("No training.csv yet. Run the gated training cell before plotting training curves.")


## 13. Final held-out test evaluation

Run only after validation-based model selection. This section loads `best_validation.pt` and calls `make_test_dataset` on the saved test manifest. It reports the repository-supported weighted classification metrics, ROC-AUC where defined, paired cosine alignment similarity, and segmentation metrics. It also records class-wise Dice/IoU and saves patch-space per-case predictions. The test set is not used for checkpoint selection. Segmentation HD95 uses unit voxel spacing in the current utility because dataset spacing is not passed through; empty-empty class Dice/IoU is 1 and HD95 is undefined.

In [ ]:
import json, numpy as np, torch
from torch.utils.data import DataLoader
from neurolens.training.train import make_test_dataset
from neurolens.training.checkpoint import load_checkpoint
from neurolens.evaluation.classification import evaluate_classification
from neurolens.evaluation.alignment import alignment_similarity
from neurolens.evaluation.segmentation import segmentation_metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

BEST_CHECKPOINT = Path(cfg.output_dir) / "best_validation.pt"
RUN_FINAL_EVALUATION = False  # Set True after validation-based model selection to unlock held-out test evaluation.
if not RUN_FINAL_EVALUATION:
    print("Held-out test evaluation is deferred. Set RUN_FINAL_EVALUATION=True only after selecting best_validation.pt.")
elif not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Best-validation checkpoint not found: {BEST_CHECKPOINT}. Train first; test evaluation will not select a model.")
else:
    test_dataset = make_test_dataset(cfg)
    test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers,
                             pin_memory=torch.cuda.is_available())
    eval_model = NeuroLens(len(cfg.modalities), cfg.base_channels, cfg.projection_dim, cfg.num_classes, cfg.classification_classes)
    eval_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    best_epoch = load_checkpoint(BEST_CHECKPOINT, eval_model, map_location=eval_device)
    eval_model.to(eval_device).eval()
    classification_result = evaluate_classification(eval_model, test_loader, eval_device)
    alignment_result = alignment_similarity(eval_model, test_loader, eval_device)
    truth_cls, pred_cls, probabilities = [], [], []
    case_seg_metrics, case_results = [], []
    class_names = {0: "background", 1: "NCR", 2: "ED", 3: "ET"}
    per_class = {c: {"dice": [], "iou": []} for c in (1, 2, 3)}
    prediction_dir = Path(cfg.output_dir) / "evaluation/per_case"; prediction_dir.mkdir(parents=True, exist_ok=True)
    with torch.inference_mode():
        for test_batch in test_loader:
            x2 = test_batch["image2d"].to(eval_device).float(); x3 = test_batch["image3d"].to(eval_device).float()
            output = eval_model(x2, x3)
            probs = output["classification_logits"].softmax(1).cpu().numpy()
            preds = probs.argmax(1)
            y = test_batch["class_label"].cpu().numpy()
            truth_cls.extend(y.tolist()); pred_cls.extend(preds.tolist()); probabilities.extend(probs.tolist())
            seg_pred = output["segmentation_logits"].argmax(1).cpu().numpy()
            seg_truth = test_batch["segmentation"].cpu().numpy()
            for i, patient_id in enumerate(test_batch["patient_id"]):
                sm = segmentation_metrics(seg_pred[i], seg_truth[i])
                case_seg_metrics.append(sm)
                for c in (1, 2, 3):
                    a, b = seg_pred[i] == c, seg_truth[i] == c
                    inter = np.logical_and(a, b).sum(); denom = a.sum() + b.sum(); union = np.logical_or(a, b).sum()
                    per_class[c]["dice"].append(float(2 * inter / denom) if denom else 1.0)
                    per_class[c]["iou"].append(float(inter / union) if union else 1.0)
                np.savez_compressed(prediction_dir / f"{patient_id}.npz", patient_id=np.asarray(patient_id),
                                    classification_true=y[i], classification_pred=preds[i], classification_probabilities=probs[i],
                                    segmentation_prediction=seg_pred[i].astype(np.uint8), segmentation_target=seg_truth[i].astype(np.uint8),
                                    axis_order=np.asarray("(D,H,W)=(Z,Y,X), cropped patch space"))
                case_results.append({"patient_id": patient_id, "classification_true": int(y[i]),
                                     "classification_pred": int(preds[i]), "classification_probabilities": probs[i].tolist(),
                                     "segmentation": sm})

    macro_seg = {metric: float(np.nanmean([r[metric] for r in case_seg_metrics])) for metric in ("dice", "iou", "hd95")}
    class_summary = {class_names[c]: {metric: float(np.mean(values)) for metric, values in per_class[c].items()} for c in per_class}
    results = {"checkpoint": str(BEST_CHECKPOINT), "selected_epoch": best_epoch,
               "test_patient_count": len(test_dataset), "classification": classification_result,
               "classification_label_meaning": {"0": "no tumour", "1": "tumour present"},
               "confusion_matrix_order": [0, 1], "segmentation_macro_foreground": macro_seg,
               "segmentation_per_class_foreground": class_summary,
               "segmentation_label_meaning": {str(k): v for k, v in class_names.items()},
               "alignment": alignment_result,
               "region_metrics": "Not emitted by the repository implementation; no WT/TC/ET aggregation is inferred here.",
               "per_case_predictions": str(prediction_dir), "cases": case_results}
    eval_dir = Path(cfg.output_dir) / "evaluation"; eval_dir.mkdir(parents=True, exist_ok=True)
    (eval_dir / "results.json").write_text(json.dumps(results, indent=2, allow_nan=True), encoding="utf-8")
    cm = confusion_matrix(truth_cls, pred_cls, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 4)); ConfusionMatrixDisplay(cm, display_labels=["no tumour", "tumour"]).plot(ax=ax, colorbar=False); fig.tight_layout(); fig.savefig(eval_dir / "confusion_matrix.png", dpi=150); plt.show()
    print(json.dumps({k: v for k, v in results.items() if k != "cases"}, indent=2, allow_nan=True))


## 14. Qualitative predictions and error review

The following example loads one saved test patch artifact, then displays a channel plane and corresponding ground truth/prediction overlay. The arrays are in cropped patch coordinates. Review errors by tumour region and tumour-presence classification; do not describe a visual example as a performance estimate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

per_case_dir = Path(cfg.output_dir) / "evaluation/per_case"
artifacts = sorted(per_case_dir.glob("*.npz"))
if not artifacts:
    print("No qualitative predictions yet; run the final evaluation cell first.")
else:
    example = np.load(artifacts[0])
    gt, pred = example["segmentation_target"], example["segmentation_prediction"]
    plane = gt.shape[0] // 2
    # The saved patch output includes labels; retrieve the matching model input from the saved test dataset.
    patient = str(example["patient_id"])
    index = next(i for i, row in enumerate(test_dataset.cases) if row[0] == patient)
    ex = test_dataset[index]
    img = ex["image3d"][0].numpy()[plane]
    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    ax[0].imshow(img, cmap="gray"); ax[0].set_title("Input t1n, central depth")
    ax[1].imshow(img, cmap="gray"); ax[1].imshow(np.ma.masked_where(gt[plane] == 0, gt[plane]), alpha=.5, vmin=1, vmax=3, cmap="viridis"); ax[1].set_title("Ground truth")
    ax[2].imshow(img, cmap="gray"); ax[2].imshow(np.ma.masked_where(pred[plane] == 0, pred[plane]), alpha=.5, vmin=1, vmax=3, cmap="viridis"); ax[2].set_title("Prediction")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()
    print("Patient:", patient, "classification true/pred:", int(example["classification_true"]), int(example["classification_pred"]))
    print("Discuss errors with the saved class-wise Dice/IoU and overlays; do not infer correctness from this single case.")


## 15. Automated experiment reporting

The report combines measured configuration, split counts, parameter count, run metadata, validation history, and final held-out results when those files exist. Missing results remain explicitly marked as not run; no values are estimated or fabricated.

In [ ]:
from pathlib import Path
import json, csv

report_dir = Path(cfg.output_dir); report_dir.mkdir(parents=True, exist_ok=True)
report_lines = ["# NeuroLens Condition A Experiment Report", "",
                "## Configuration", "```json", json.dumps(asdict(cfg), indent=2, default=list), "```", "",
                "## Data and split", f"- BraTS root: `{cfg.brats_root}`",
                f"- Saved split counts: `{json.dumps(split_counts)}`",
                f"- Classification target: binary tumour presence from the cropped segmentation.",
                f"- Segmentation mapping: 0 background, 1 NCR, 2 ED, 3 ET (source label 4 mapped to 3).",
                "- BRISC remains independent 2D data; no BRISC/BraTS positive pairs are created.", "",
                "## Model and method", f"- Trainable parameters: `{sum(p.numel() for p in architecture_model.parameters() if p.requires_grad) if 'architecture_model' in globals() else (sum(p.numel() for p in model.parameters() if p.requires_grad) if 'model' in globals() else 'model not instantiated in this session')}`",
                "- Condition: A; architecture from `neurolens.models.multitask.NeuroLens`.",
                "- Optimizer: AdamW; configured LR and weight decay recorded above.",
                "- Scheduler: none in current implementation.", "",
                "## Training history"]
csv_path = report_dir / "training.csv"
if csv_path.is_file():
    with csv_path.open(newline="", encoding="utf-8") as f: hist = list(csv.DictReader(f))
    report_lines.append(f"- Epoch records: {len(hist)}")
    if hist: best_row = min(hist, key=lambda row: float(row["val_total"])); report_lines.append(f"- Best observed validation total in CSV: {float(best_row['val_total']):.6f} at epoch {int(float(best_row['epoch']))}")
else: report_lines.append("- Not run; no training.csv exists.")
metadata_path = report_dir / "run_metadata.json"
if metadata_path.is_file():
    report_lines.extend(["", "## Runtime", "```json", json.dumps(json.loads(metadata_path.read_text()), indent=2), "```"])
else: report_lines.append("- Runtime metadata: not available; training has not completed in this run.")
results_path = report_dir / "evaluation/results.json"
if results_path.is_file():
    measured = json.loads(results_path.read_text())
    summary = {k: measured.get(k) for k in ("selected_epoch", "test_patient_count", "classification", "segmentation_macro_foreground", "segmentation_per_class_foreground", "alignment")}
    report_lines.extend(["", "## Held-out test metrics", "```json", json.dumps(summary, indent=2, allow_nan=True), "```"])
else: report_lines.extend(["", "## Held-out test metrics", "Not run: evaluate the best-validation checkpoint in Section 13 after model selection."])
report_lines.extend(["", "## Checkpoints", f"- Latest: `{report_dir / 'latest.pt'}`", f"- Best validation: `{report_dir / 'best_validation.pt'}`",
                     "", "## Curves and qualitative artifacts", f"- Plots: `{report_dir / 'plots'}`", f"- Per-case patch predictions: `{report_dir / 'evaluation/per_case'}`",
                     "", "## Interpretation template", "### Measured findings", "Record only values emitted by this run and discuss split-level uncertainty.",
                     "### Limitations", "Discuss the 371-patient cohort, stochastic training slice selection, crop-space output, and any incomplete runs.",
                     "### Hypotheses and future work", "Clearly label proposed experiments; do not present them as measured results."])
report_path = report_dir / "experiment_report.md"
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
print("Report saved:", report_path)
print(report_path.read_text(encoding="utf-8"))


## 16. Research conclusions template

Complete this section after reviewing measured outputs. Separate observations from interpretations and plans.

- **Baseline performance (measured):** report selected validation epoch and held-out test metrics from `evaluation/results.json`.
- **Classification:** describe binary tumour-presence performance and confusion matrix; do not call it subtype classification.
- **Segmentation:** report foreground macro and NCR/ED/ET Dice/IoU, including empty-region behavior.
- **Cross-dimensional alignment:** report paired cosine similarity as descriptive evidence; Condition A does not optimize alignment losses.
- **Limitations:** note cohort size, crop-space inference, slice sampling, and any compute or data limitations.
- **Potential improvements (hypotheses):** propose ablations or preprocessing changes without implying they have been tested.
- **Future experiments:** define conditions, splits, and metrics before looking at test results.

### Current status

Notebook construction performs static validation only. It does not execute the real-data audit, train the model, evaluate test data, or claim model results. Execute those notebook cells in Colab with the expected Drive files and GPU to produce measured outputs.